## Credit Stress Index Analysis V5

In [1]:
import os
import datetime as dt

from dotenv import load_dotenv
from pathlib import Path

import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown
from Modules.compute_fibonacci_extension import ComputeFibonacciExtension

In [2]:
# FRED API key for accessing Federal Reserve Economic Data
FRED_API_KEY = os.getenv("FRED_API_KEY")
# Output directory for storing downloaded data
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()
# Fibonacciエクステンションを計算するクラスのインスタンスを作成
fibonacci_extension = ComputeFibonacciExtension()

In [3]:
# Date range for fetching API s
start = '2007-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)

#### SJBの時系列データを取得

In [4]:
sjb = 'SJB'

response = request_api.update_stock_timeseries_data(
    code=sjb,
    market=None,
    start=start,
    end=end
)
response

{'result': True}

In [5]:
sjb_df = request_api.get_stock_time_series_data(
    code=sjb,
    market=None,
    start=start,
    end=end
)
sjb_df["date"] = pd.to_datetime(sjb_df["date"])
sjb_df

取得件数: 3912


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1165760,SJB,None,2011-03-22,33.406315,33.531590,33.406315,33.531590,7700,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1165761,SJB,None,2011-03-23,33.406315,33.539940,33.406315,33.539940,5900,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1165762,SJB,None,2011-03-24,33.347858,33.431373,33.347858,33.431373,1700,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1165763,SJB,None,2011-03-25,33.381256,33.397960,33.364553,33.397960,3600,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1165764,SJB,None,2011-03-28,33.347858,33.423023,33.339505,33.423023,99800,33.464777,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3907,1169667,SJB,None,2026-10-05,15.480000,15.520000,15.480000,15.510000,128500,15.470000,...,15.075001,61.081704,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3908,1169668,SJB,None,2026-10-06,15.420000,15.450000,15.410000,15.430000,235700,15.470000,...,15.090617,65.837104,True,NaN,NaN,NaN,NaN,NaN,93.196581,False
3909,1169669,SJB,None,2026-10-07,15.450000,15.490000,15.430000,15.440000,85700,15.468000,...,15.105892,70.472979,True,NaN,NaN,NaN,NaN,NaN,NaN,False
3910,1169670,SJB,None,2026-10-08,15.450000,15.490000,15.450000,15.450000,358900,15.456000,...,15.120783,74.494152,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### DX-Yの時系列データを取得

In [6]:
dxy = 'DX-Y'
market = 'NYB'

response = request_api.update_stock_timeseries_data(
    code=dxy,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [7]:
dx_df = request_api.get_stock_time_series_data(
    code=dxy,
    market=market,
    start=start,
    end=end
)
dx_df["date"] = pd.to_datetime(dx_df["date"])
dx_df

取得件数: 5185


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1091754,DX-Y,NYB,2006-03-07,90.769997,90.860001,89.860001,89.910004,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1091755,DX-Y,NYB,2006-03-08,90.620003,90.800003,90.430000,90.709999,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1091756,DX-Y,NYB,2006-03-09,90.750000,90.790001,90.379997,90.599998,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1091757,DX-Y,NYB,2006-03-10,90.849998,91.160004,90.589996,90.779999,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1091758,DX-Y,NYB,2006-03-13,90.540001,90.870003,90.519997,90.830002,0,90.566000,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5180,1187717,DX-Y,NYB,2026-10-05,102.169998,102.540001,101.860001,101.919998,0,101.602000,...,99.174593,8.025271,True,NaN,NaN,NaN,NaN,NaN,NaN,False
5181,1187718,DX-Y,NYB,2026-10-06,101.830002,102.290001,101.750000,102.139999,0,101.792000,...,99.206891,16.750619,True,NaN,NaN,NaN,NaN,NaN,NaN,False
5182,1187719,DX-Y,NYB,2026-10-07,102.239998,102.500000,101.889999,101.889999,0,101.887999,...,99.275607,26.278494,True,NaN,NaN,NaN,NaN,NaN,91.82906,False
5183,1187720,DX-Y,NYB,2026-10-08,102.139999,102.470001,102.029999,102.230003,0,102.039999,...,99.330177,35.055067,True,NaN,NaN,NaN,NaN,NaN,NaN,False


#### HYGの時系列データを取得

In [8]:
hyg = 'HYG'

response = request_api.update_stock_timeseries_data(
    code=hyg,
    market=None,
    start=start,
    end=end
)
response

{'result': True}

In [9]:
hyg_df = request_api.get_stock_time_series_data(
    code=hyg,
    market=None,
    start=start,
    end=end
)
hyg_df["date"] = pd.to_datetime(hyg_df["date"])
hyg_df

取得件数: 4907


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1169672,HYG,None,2007-04-11,31.084785,31.096701,31.084785,31.084785,1900,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1169673,HYG,None,2007-04-12,31.105623,31.105623,31.084771,31.105623,6100,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1169674,HYG,None,2007-04-13,31.049049,31.126499,31.049049,31.096711,3400,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1169675,HYG,None,2007-04-16,31.037115,31.060945,31.025199,31.060945,7600,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1169676,HYG,None,2007-04-17,31.022232,31.099684,31.022232,31.025212,28100,31.074655,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4902,1174574,HYG,None,2026-10-05,76.980003,77.019997,76.809998,76.889999,39142800,77.050616,...,77.430472,-36.032613,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4903,1174575,HYG,None,2026-10-06,77.269997,77.320000,77.139999,77.150002,47352500,77.027371,...,77.356343,-44.040809,False,NaN,NaN,NaN,NaN,NaN,NaN,True
4904,1174576,HYG,None,2026-10-07,77.180000,77.209999,76.930000,77.000000,59309100,77.002000,...,77.275755,-51.707504,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4905,1174577,HYG,None,2026-10-08,77.139999,77.180000,76.980003,77.070000,44872800,77.066000,...,77.207912,-58.375309,False,NaN,NaN,NaN,NaN,NaN,NaN,True


#### TLTの時系列データを取得

In [10]:
tlt = 'TLT'

response = request_api.update_stock_timeseries_data(
    code=tlt,
    market=None,
    start=start,
    end=end
)
response

{'result': True}

In [11]:
tlt_df = request_api.get_stock_time_series_data(
    code=tlt,
    market=None,
    start=start,
    end=end
)
tlt_df["date"] = pd.to_datetime(tlt_df["date"])
tlt_df

取得件数: 5182


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1175486,TLT,None,2006-03-07,45.815853,45.888240,45.660736,45.707273,1692900,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1175487,TLT,None,2006-03-08,45.800343,45.955455,45.733123,45.795171,1184700,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1175488,TLT,None,2006-03-09,45.815853,45.846875,45.640054,45.789999,282100,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1175489,TLT,None,2006-03-10,45.645233,45.743474,45.505631,45.665916,1086900,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1175490,TLT,None,2006-03-13,45.609039,45.640061,45.505631,45.572846,1740100,45.706241,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5177,1180663,TLT,None,2026-10-05,77.110001,77.260002,76.690002,77.160004,49106700,77.618303,...,78.701412,-72.457953,False,NaN,NaN,NaN,NaN,NaN,NaN,False
5178,1180664,TLT,None,2026-10-06,77.279999,77.580002,77.070000,77.169998,31085900,77.411296,...,78.433292,-74.293520,False,NaN,NaN,NaN,NaN,-85.0,NaN,False
5179,1180665,TLT,None,2026-10-07,77.150002,77.320000,76.430000,76.459999,40200000,77.134001,...,78.110772,-75.147272,False,NaN,NaN,NaN,NaN,NaN,NaN,False
5180,1180666,TLT,None,2026-10-08,77.870003,77.910004,77.070000,77.190002,49484200,77.184001,...,77.891491,-75.078972,False,NaN,NaN,NaN,NaN,NaN,NaN,True


#### FREDの月次データを取得

In [12]:
fred_url = "https://api.stlouisfed.org/fred/series/observations"
series_mapping = {
    "DFF": "effective_federal_funds_rate",          # 実効FF金利
    "DFII10": "real_10y",                           # 10年TIPS実質金利
    "FEDFUNDS": "ff_rate",                          # FF金利
    "DFEDTARU": "maximum_interest_rate",            # 上限金利
    "DFEDTARL": "minimum_interest_rate",            # 下限金利
    "DGS2": "us_treasury_2y",                       # 2年米国債
    "DGS5": "us_treasury_5y",                       # 5年米国債
    "DGS10": "us_treasury_10y",                     # 10年米国債
    "DGS30": "us_treasury_30y",                     # 30年米国債
    "M1SL": "m1_money_stock",                       # M1マネーストック
    "M2SL": "m2_money_stock",                       # M2マネーストック
    "BAMLH0A0HYM2": "hy_oas",                       # ハイイールドOAS
    "BAMLC0A4CBBB": "bbb_oas",                      # BBB OAS
    "NFCI": "national_financial_conditions_index",  # 全国金融状況指数
    "VIXCLS": "vix",                                # VIX指数
    "WALCL": "fed_balance_sheet",                   # FRBバランスシート
}
observation_start = "2007-01-01"
observation_end = end
base_fred_params = {
    "api_key": FRED_API_KEY,
    "file_type": "json",
    "observation_start": observation_start,
    "observation_end": observation_end,
    "frequency": "m",
    "aggregation_method": "avg",
}
fred_dfs = []
for series_id, series_name in series_mapping.items():
    fred_params = base_fred_params.copy()
    fred_params["series_id"] = series_id
    fred_res = requests.get(fred_url, params=fred_params)
    fred_df = pd.DataFrame(fred_res.json()["observations"]).assign(**{
        "date": lambda df: pd.to_datetime(df["date"]),
        "value": lambda df: pd.to_numeric(df["value"], errors="coerce"),
        "interest_rate_type": series_name,
    })
    fred_dfs.append(fred_df)
    time.sleep(5)
    
us_treasury_df = pd.concat(fred_dfs)
us_treasury_df

,realtime_start,realtime_end,date,value,interest_rate_type
0,2026-10-11,2026-10-11,2007-01-01,5.25,effective_federal_funds_rate
1,2026-10-11,2026-10-11,2007-02-01,5.26,effective_federal_funds_rate
2,2026-10-11,2026-10-11,2007-03-01,5.26,effective_federal_funds_rate
3,2026-10-11,2026-10-11,2007-04-01,5.25,effective_federal_funds_rate
4,2026-10-11,2026-10-11,2007-05-01,5.25,effective_federal_funds_rate
...,...,...,...,...,...
233,2026-10-12,2026-10-12,2026-06-01,6727240.00,fed_balance_sheet
234,2026-10-12,2026-10-12,2026-07-01,6737754.00,fed_balance_sheet
235,2026-10-12,2026-10-12,2026-08-01,6746283.00,fed_balance_sheet
236,2026-10-12,2026-10-12,2026-09-01,6743021.00,fed_balance_sheet


### CSI v5 (Credit Stress Index v5)

#### 0. ETF・FREDデータの結合

#### 1. 月次リサンプリング

#### 2. 基本特徴量の作成

##### 2-1. 金利関連特徴量

##### 2-2. 流動性関連特徴量

##### 2-3. クレジットスプレッド特徴量

##### 2-4. マーケットストレス特徴量

##### 2-5. ETF関連特徴量

#### 3. 目的変数の作成

##### 3-1. HYG Forward Return

##### 3-2. HYG Future Drawdown

#### 4. データクレンジング

##### 4-1. 欠損値処理

##### 4-2. サンプル期間統一

#### 5. 単独因子相関分析

##### 5-1. Pearson Correlation

##### 5-2. Spearman Correlation

##### 5-3. Rank IC

#### 6. Lead / Lag 分析

##### 6-1. Forward ReturnとのLead/Lag分析

##### 6-2. DrawdownとのLead/Lag分析

#### 7. 共線性分析

##### 7-1. 相関行列

##### 7-2. ヒートマップ

##### 7-3. VIF分析

#### 8. CSI v5候補因子の選定

##### 8-1. 相関順位

##### 8-2. Lead/Lag順位

##### 8-3. 採用因子決定

#### 9. CSI v5 Equal Weight 作成

#### 10. CSI v5 Correlation Weight 作成

#### 11. CSI v5 PCA Weight 作成

#### 12. CSI v5 Backtest

##### 12-1. Pearson

##### 12-2. Spearman

##### 12-3. Rank IC

##### 12-4. Decile Analysis

#### 13. Drawdown予測能力評価

##### 13-1. Pearson

##### 13-2. Spearman

##### 13-3. Rank IC

#### 14. 可視化

##### 14-1. CSI v5

##### 14-2. HYG

##### 14-3. CSI vs HYG

##### 14-4. CSI vs Drawdown

#### 15. V4との比較

##### 15-1. 相関比較

##### 15-2. Rank IC比較

##### 15-3. Drawdown予測比較

#### 16. 最終評価

##### 採用因子一覧

##### CSI v5計算式

##### CSI v4比較

##### 今後の改善候補